<a href="https://colab.research.google.com/github/Hamerson-jhoel/Procesamiento-de-Lengaje-Natural/blob/main/Clase_8_04_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [17]:
# Carga del dataset desde la URL corregida
url = "https://raw.githubusercontent.com/UN-GCPDS/ProcesamientoLenguajeNatural/refs/heads/main/Insumos/SMSSpamCollection.txt"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Visualización de la distribución de las categorías
print("Distribución de clases:")
print(df['label'].value_counts())

Distribución de clases:
label
ham     4825
spam     747
Name: count, dtype: int64


In [18]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

def tokenize(text):
  text= re.sub(r'[^\w\s]', '' , text.lower())
  return text.split()

all_words = []
for msg in df['message']:
  all_words.extend(tokenize(msg))

counts = Counter(all_words)

frequent_words = [word for word, count in counts.items() if count > 1]
#devuelve la palabra y cuantas veces se repite la palabra
vocab ={word: i+2 for i, word in enumerate(frequent_words)}
#rellena con ceros la oracion , si esta en 1 lo deja igual, si esta en 1 significa que hay uhna palabra , y con el cero lo que hace es esto para pasarle el mismo largo a una red,
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

print(len(vocab), max(vocab.values()))


4366 4365


In [19]:
#vamos a utilizar el 95% de los mensajes sin recortarlos demasiado

df['msg_len'] = df['message'].apply(lambda x: len(tokenize(x)))
max_length = int(df['msg_len'].quantile(0.95))

print(max_length)

32


In [20]:
# vamos a crear una funcion que convierte texto en una lista de indices de longitud fija

def encode_and_pad(text, vocab, max_len):
  tokens = tokenize(text)
  encoded =[vocab.get(w,1) for w in tokens]


  if len(encoded) < max_len:
    encoded += [0] * (max_len - len(encoded))
  else:
    encoded = encoded[:max_len]
  return encoded

In [21]:
#vamos a crear un x , las muestras para que el modelo me de las caracteristicas

x = np.array([encode_and_pad(m, vocab, max_length) for m in df['message']])
y = df['label'].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape)

(4457, 32)


In [22]:
# esta clase sirve para
class SpamDataset(Dataset):
  def __init__(self, x, y):
    self.x = torch.tensor(x, dtype=torch.long)
    self.y = torch.tensor(y, dtype=torch.float32)

  def __len__(self):
    return len(self.y)

  def __getitem__(self, idx):
    return self.x[idx], self.y[idx]

In [23]:
#juntar 32 oracionse para que me queden en grupo, dataloader revuelve y agrupa, y en cada grupo
train_loader = DataLoader(SpamDataset(x_train, y_train), batch_size=32,
shuffle=True)
test_loader = DataLoader(SpamDataset(x_test, y_test), batch_size=32)

sample_x, sample_y = next(iter(train_loader))
print(sample_x.shape, sample_y.shape)


torch.Size([32, 32]) torch.Size([32])


In [24]:
#red recuerente , las redes son perezozas,
class SimpleRNNModel(nn.Module):

  def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
    super(SimpleRNNModel, self).__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
#dropurt que explore mas caminos y no solo en una, para que no se vaya por #el camino facil, y que se sentre en las caracteristicas de cada muestra
    self.dropout = nn.Dropout(0.5)

    self.fc = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):

    embedded = self.embedding(x)

    out, hidden = self.rnn(embedded)

    final_state = self.dropout(hidden.squeeze(0)) # 1,0 Hd -> 0,Hd


    return self.fc(final_state)

In [25]:
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
model = SimpleRNNModel  (len(vocab), EMBEDDING_DIM, HIDDEN_DIM, 1).to(device)

In [26]:
#componenetes que nos ayudan aa entrenar, los vamos a crear
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # le pasamos los parametrso del modelo, y la tasa de aprendizaje sera del 0.001


In [27]:
model

SimpleRNNModel(
  (embedding): Embedding(4366, 100)
  (rnn): RNN(100, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [34]:
def train_model(model, train_loader, optimizer, criterion, epochs= 5):
  model.train()
  for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for batch_x, batch_y in train_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      optimizer.zero_grad()

      predictions = model(batch_x).squeeze(1)
      loss = criterion(predictions, batch_y)
      loss.backward()
      optimizer.step()
      epoch_loss += loss

      preds = torch.round(torch.sigmoid(predictions))
      correct += (preds == batch_y).sum().item()
      total += batch_y.size(0)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}, Accuracy: {correct/total:.4f}")
  return model

In [35]:
model = train_model(model, train_loader, optimizer, criterion)

Epoch 1/5, Loss: 0.4075, Accuracy: 0.8640
Epoch 2/5, Loss: 0.4025, Accuracy: 0.8658
Epoch 3/5, Loss: 0.3989, Accuracy: 0.8663
Epoch 4/5, Loss: 0.3886, Accuracy: 0.8670
Epoch 5/5, Loss: 0.3910, Accuracy: 0.8611


In [ ]:
#modelo MEMORIA LARGA corta, LSPM, comparte informacion osea las notas del compañero pero le comparte lo que crea que sea necesarfio al otro compañero, direfrente de la redes recurrentes que solo comparte informacion , no las notas de cada compañero ,


#es mucho mejor que el anterior es importnte porque se comparte la informacion la una a la otra se comparte la informacion y aparte el ultimo compañero ve informacion de los demas compañeros

#miran la informacion completa

In [36]:
class LSTMModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMModel, self).__init__()

        self. embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded)
        final_state = self.dropout(hidden[-1]) # 1,0 Hd -> 0,Hd
        return self.fc(final_state)


In [37]:
lstm_model = LSTMModel(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, 1).to(device)
optimizer_lstm= optim.Adam(lstm_model.parameters(), lr=0.001)


In [38]:
lstm_model = train_model(lstm_model, train_loader, optimizer_lstm, criterion)

Epoch 1/5, Loss: 0.3241, Accuracy: 0.8880
Epoch 2/5, Loss: 0.1091, Accuracy: 0.9702
Epoch 3/5, Loss: 0.0599, Accuracy: 0.9850
Epoch 4/5, Loss: 0.0484, Accuracy: 0.9881
Epoch 5/5, Loss: 0.0371, Accuracy: 0.9919


#clase que nos ayude a evaluar nuestros modelos

In [39]:
def evaluate_model(model, loader):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for batch_x, batch_y in loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      outputs = model(batch_x).squeeze(1)
      preds = torch.round(torch.sigmoid(outputs))
      correct += (preds == batch_y).sum().item()
      total += batch_y.size(0)

  print(f"Accuracy in test partition: {correct/total:.4f}")



In [40]:
evaluate_model(model, test_loader)
evaluate_model(lstm_model, test_loader)

Accuracy in test partition: 0.6278
Accuracy in test partition: 0.9865


In [41]:
def predict_sms(message, model, vocab, max_len):
  model.eval()
  encoded = encode_and_pad(message, vocab, max_len)
  tensor_input = torch.tensor([encoded], dtype=torch.long).to(device)

  with torch.no_grad():
    output = model(tensor_input).squeeze(1)

    prob = torch.sigmoid(output).item()

  result = "SPAM" if prob > 0.5 else "HAM"

  print(f"Mensaje: {message}")
  print(f"Probabilidad: {prob:.4f}")
  print(f"Resultado: {result}")


In [42]:
predict_sms(" WINNERI you have won a 1000 ash prize, text 'CLAIN to 888", lstm_model,vocab, max_length)
predict_sms(" WINNERI you have won a 1000 ash prize, text 'CLAIN to 888", model,vocab, max_length)

Mensaje:  WINNERI you have won a 1000 ash prize, text 'CLAIN to 888
Probabilidad: 0.8940
Resultado: SPAM
Mensaje:  WINNERI you have won a 1000 ash prize, text 'CLAIN to 888
Probabilidad: 0.7046
Resultado: SPAM


#HACERLO CON GRU, EN redes recurrente

In [43]:
#red recuerente , las redes son perezozas,
class SimpleGRUModel(nn.Module):

  def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
    super(SimpleGRUModel, self).__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
#dropurt que explore mas caminos y no solo en una, para que no se vaya por #el camino facil, y que se sentre en las caracteristicas de cada muestra
    self.dropout = nn.Dropout(0.5)

    self.fc = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):

    embedded = self.embedding(x)

    out, hidden = self.gru(embedded)

    final_state = self.dropout(hidden.squeeze(0)) # 1,0 Hd -> 0,Hd


    return self.fc(final_state)

In [47]:
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
model = SimpleGRUModel  (len(vocab), EMBEDDING_DIM, HIDDEN_DIM, 1).to(device)

In [48]:
#componenetes que nos ayudan aa entrenar, los vamos a crear
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # le pasamos los parametrso del modelo, y la tasa de aprendizaje sera del 0.001


In [49]:
model

SimpleGRUModel(
  (embedding): Embedding(4366, 100)
  (gru): GRU(100, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [50]:
def train_model(model, train_loader, optimizer, criterion, epochs= 5):
  model.train()
  for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for batch_x, batch_y in train_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      optimizer.zero_grad()

      predictions = model(batch_x).squeeze(1)
      loss = criterion(predictions, batch_y)
      loss.backward()
      optimizer.step()
      epoch_loss += loss

      preds = torch.round(torch.sigmoid(predictions))
      correct += (preds == batch_y).sum().item()
      total += batch_y.size(0)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}, Accuracy: {correct/total:.4f}")
  return model

In [51]:
model = train_model(model, train_loader, optimizer, criterion)

Epoch 1/5, Loss: 0.2630, Accuracy: 0.9013
Epoch 2/5, Loss: 0.0816, Accuracy: 0.9764
Epoch 3/5, Loss: 0.0507, Accuracy: 0.9852
Epoch 4/5, Loss: 0.0326, Accuracy: 0.9906
Epoch 5/5, Loss: 0.0176, Accuracy: 0.9955


### Evaluating the GRU Model

In [52]:
evaluate_model(model, test_loader)

Accuracy in test partition: 0.9830
